# QLoRA training (Colab)

Run this on a Colab GPU runtime (Runtime > Change runtime type > GPU).

This notebook runs the **real 4-bit QLoRA** training path. It reuses the exact same
code as the repo (`train/qlora_train.py`) so the Colab run and the local CPU smoke
test can never drift: when CUDA is present, `qlora_train.py` loads the model in 4-bit
with Unsloth and trains with TRL's `SFTTrainer`; on CPU it falls back to plain `peft`
LoRA (that fallback is what proved the plumbing on Day 2).

Prereqs (Day 2, done): Behavior Spec written, data pipeline in `data/`, eval harness
in `eval/`, and the end-to-end smoke test passing. Day 3 uses this for the first real
training run on the v1 dataset.

## 1. Install environment

In [3]:
!pip install -q unsloth trl datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

## 2. Load base model and confirm GPU inference works

In [4]:
from unsloth import FastLanguageModel

MODEL_ID = "Qwen/Qwen3-4B"  # tune target upgraded 0.6B -> 4B to clear the accuracy floor
                            # (0.6B was capacity-bound: litmus accuracy 5/12). 4B is the CAP.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,   # QLoRA 4-bit: ~2.2GB weights; full run peaks ~6-8GB, fits a free T4 (16GB)
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-1): 2 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm

In [5]:
messages = [{"role": "user", "content": "In one sentence, what is a QLoRA fine-tune?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

<think>
Okay, the user is asking for a one-sentence definition of a QLoRA fine-tune. I need to make sure I get the key components right. QLoRA is a technique, right? It's related to quantization, low-rank adaptation, and fine-tuning. So, the main points are quantization, low-rank adaptation, and the purpose is to fine-tune large models efficiently. Let me check if I have all the parts. Quantization reduces


## 3. Train on the generated dataset (real QLoRA)

Clone the repo so the notebook runs the *same* code and dataset as everything else,
then invoke the training module. On a GPU runtime this takes the Unsloth 4-bit QLoRA
path automatically.

In [7]:
# Get the repo so the notebook runs the *same* code + dataset as everything else.
%cd /content
![ -d SmallLearningModel ] || git clone https://github.com/Elitelord/SmallLearningModel.git
%cd SmallLearningModel
!git pull  # make sure you have the latest committed dataset + code

# QLoRA run on the v2 (round-2) gold dataset (Qwen3-4B). Unsloth 4-bit + TRL SFTTrainer on GPU.
# --data is the tighter-centered v2 set (225 ex, FK ~4.4); must be COMMITTED & PUSHED
# (Colab clones from GitHub, so uncommitted local files are NOT visible here).
# v1 was data/v4/gold_v4_final.jsonl / --adapter-name v4 (kept for comparison).
!python -m train.qlora_train \
    --data data/v4/gold_v4_r2_final.jsonl \
    --adapter-name v4r2 \
    --epochs 3 \
    --lora-r 16 --lora-alpha 32 \
    --batch-size 8 --grad-accum 2 --lr 2e-4

/content
Cloning into 'SmallLearningModel'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 232 (delta 109), reused 183 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (232/232), 1.26 MiB | 19.56 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/SmallLearningModel
Already up to date.
Loaded 225 training records from data/v4/gold_v4_r2_final.jsonl
CUDA detected -> Unsloth 4-bit QLoRA path
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and

## 4. Base-vs-tuned eval

Score the base model and the freshly trained adapter on the held-out concepts with the
identical minimal prompt (`Explain: {concept}`), reusing the litmus **v4** readability
gate (whole-passage FK 3–6 AND ARI 3–7) + the 0/1/2 accuracy judge. Both sides are
served with `enable_thinking=False`, matching training.

The cell below installs the eval deps and runs `--no-judge` first (FK/ARI deltas, no
API). To add the accuracy delta, set the gateway creds (via a Colab Secret) and pass
`--judge openai-group/gpt-4.1` as shown in the commented lines.

In [9]:
# Eval needs the readability scorer (textstat); the accuracy judge also needs openai.
!pip install -q textstat openai

# Readability-only run: FK/ARI deltas on held-out concepts, no API. Adapter name
# MUST match the training cell (v4r2).
!python -m eval.base_vs_tuned --adapter train/adapters/v4r2 --limit 12 --no-judge

# ACCURACY: fastest path is to score the saved generations LOCALLY afterward — the
# results JSON stores base_text/tuned_text, so you judge them via the gateway from
# your own machine (~24 calls) without any Colab gateway setup.
# Or, to judge here, set gateway creds (Colab Secret — do NOT hardcode; this notebook
# is pushed to GitHub) and pass the slug:
#   import os
#   from google.colab import userdata
#   os.environ["OPENAI_API_KEY"]  = userdata.get("TFY_KEY")
#   os.environ["OPENAI_BASE_URL"] = "https://tfy.promptlens.trilogy.com"
#   !python -m eval.base_vs_tuned --adapter train/adapters/v4r2 --limit 12 --judge openai-group/gpt-4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 87.7 MB/s eta 0:00:00
=== base vs tuned | 12 held-out concepts | judge=off ===

Generating BASE outputs...
config.json: 100% 726/726 [00:00<00:00, 3.09MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 23.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 117MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 161MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:01<00:00, 8.16MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
model.safetensors.index.json: 100% 32.8k/32.8k [00:00<00:00, 77.1MB/s]
Fetching 3 files: 100% 3/3 [07:26<00:00, 148.99s/it]
Download complete: 100% 8.04G/8.04G [07:27<00:00, 18.0MB/s]
Loading weights: 100% 398/398 [00:32<00:00, 12.26it/s]
generation_config.json: 100% 239/239 [00:00<00:00, 1.19MB/s]
  [1/12